# Measurement realism: retained results

> **CONTROLLED SCENARIO — NOT A DATASET RELEASE.** The matched learner-facing
> bank failed its preregistered freeze gate, so the A–D, structured-error, and
> schedule worlds use a deterministic **content-free controlled instrument**.
> They establish structural sensitivity only; they do not establish learner-facing
> measurement validity or platform plausibility.
>
> **NO HUMAN VALIDATION.** Item, KC, bank, and dialogue judgments shown here are
> automated stress tests—not judgments from learners, teachers, measurement
> experts, or platform practitioners. No new dataset is released by this notebook,
> and the frozen `grammar_kt_full_v1` reference is neither opened nor modified.

This executable notebook is an offline, read-only view over seven compact retained
JSON analyses. It never reads learner-response archives, oracle trajectories,
prediction rows, or raw model-call archives, and it makes no network/model calls.

In [1]:
EVIDENCE_ROOT = "experiments/measurement_realism"
EVIDENCE_ROOT

'experiments/measurement_realism'

In [2]:
from pathlib import Path
import hashlib
import json

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

ROOT_CANDIDATES = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
ROOT = next(path for path in ROOT_CANDIDATES if (path / "pyproject.toml").is_file())
pd.set_option("display.max_colwidth", 140)
plt.style.use("seaborn-v0_8-whitegrid")

SOURCE_PATHS = {
    "controlled_worlds": "experiments/measurement_realism/worlds/controlled_instrument_v1/synthesis/results.json",
    "cross_audit": "experiments/measurement_realism/audits/platform_audit_synthesis.json",
    "dialogue_continuum": "experiments/measurement_realism/dialogue_pilot_live_v1/analysis.json",
    "kc_induction": "experiments/measurement_realism/kc_induction_v1/results.json",
    "matched_bank": "experiments/measurement_realism/design/bank_protocol/runs/matched_bank_v0_2_20260830/analysis/failure_analysis.json",
    "policy_recovery": "experiments/measurement_realism/worlds/controlled_instrument_v1/policy_recovery_v1/results/results.json",
    "strict_audit": "experiments/measurement_realism/audits/item_audit/summary.json"
}
EXPECTED_SOURCE_SHA256 = {
    "controlled_worlds": "55ac72dfdaf739597451e5766edb399690b780bbfa9499474c9142cf919e844a",
    "cross_audit": "ba2bbd8b454db0e54059b940cc77309a3921786ec19ae43c4f50507366189094",
    "dialogue_continuum": "5d2538e7866855782f92fe0c946bfcfb714463ae60f8879f01135f2459e797ef",
    "kc_induction": "d535a727b5c3cf87038bfef4c183824469a17614113733658a1dbae0d20bd376",
    "matched_bank": "1761db11853163421cd83cb9b4410f00ec887a2e0f574e1578c474eba203b7c3",
    "policy_recovery": "29702c895ae9ba34cd0e1313514b23694572d3ca60b9629923b4713c5340a5c6",
    "strict_audit": "8bf914158588dd6952730496e609ecb966e5278311de5476649cef6ff41ea419"
}

def sha256_file(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def load_json(name):
    path = ROOT / SOURCE_PATHS[name]
    observed = sha256_file(path)
    assert observed == EXPECTED_SOURCE_SHA256[name], (name, observed)
    return json.loads(path.read_text(encoding="utf-8"))

ARTIFACTS = {name: load_json(name) for name in SOURCE_PATHS}
strict = ARTIFACTS["strict_audit"]
cross = ARTIFACTS["cross_audit"]
kc = ARTIFACTS["kc_induction"]
bank = ARTIFACTS["matched_bank"]
worlds = ARTIFACTS["controlled_worlds"]
policy = ARTIFACTS["policy_recovery"]
dialogue = ARTIFACTS["dialogue_continuum"]

# Fail closed if any scientific boundary changes.
assert strict["scope"]["human_or_learner_evidence"] is False
assert cross["evidence_boundary"]["full_v1_mutated"] is False
assert cross["evidence_boundary"]["human_or_expert_gold"] is False
assert kc["evidence_boundary"]["human_or_expert_gold"] is False
assert bank["status"] == "FAILED_PREREGISTERED_BANK_FREEZE_GATE"
assert bank["release_gate_failure"]["freeze_permitted"] is False
assert worlds["controlled_scenario"] is True
assert worlds["content_free_instrument"] is True
assert worlds["release_eligible"] is False
assert worlds["claim_boundary"]["permitted"] == "controlled_structural_sensitivity_only"
assert worlds["claim_boundary"]["learner_facing_measurement_validity"] == "NOT_ASSESSED"
assert worlds["claim_boundary"]["platform_plausibility"] == "NOT_ASSESSED"
assert policy["controlled_scenario"] is True and policy["release_eligible"] is False
assert dialogue["evidence_boundary"]["scalar_realism_score_computed"] is False

integrity_rows = [
    {"artifact": name, "bytes": (ROOT / path).stat().st_size, "sha256": EXPECTED_SOURCE_SHA256[name]}
    for name, path in SOURCE_PATHS.items()
]
display(Markdown("**Offline integrity check: PASS — seven exact compact artifacts matched.**"))
display(pd.DataFrame(integrity_rows))

**Offline integrity check: PASS — seven exact compact artifacts matched.**

,artifact,bytes,sha256
0,controlled_worlds,235764,55ac72dfdaf739597451e5766edb399690b780bbfa9499474c9142cf919e844a
1,cross_audit,164265,ba2bbd8b454db0e54059b940cc77309a3921786ec19ae43c4f50507366189094
2,dialogue_continuum,105198,5d2538e7866855782f92fe0c946bfcfb714463ae60f8879f01135f2459e797ef
3,kc_induction,7725,d535a727b5c3cf87038bfef4c183824469a17614113733658a1dbae0d20bd376
4,matched_bank,292609,1761db11853163421cd83cb9b4410f00ec887a2e0f574e1578c474eba203b7c3
5,policy_recovery,45844,29702c895ae9ba34cd0e1313514b23694572d3ca60b9629923b4713c5340a5c6
6,strict_audit,22999,8bf914158588dd6952730496e609ecb966e5278311de5476649cef6ff41ea419


## 1. Platform-facing audit of the frozen 113-item bank

The strict census asks whether each stored task is comprehensible, answerable,
pedagogically and platform plausible, and diagnostically aligned. A second audit
used four role-specific automated critics. Percentages below describe these
automated judgments only; they are not estimates of human usability.

In [3]:
disposition_order = [
    "usable_as_stored",
    "minor_ui_or_context_change",
    "technically_valid_but_artificial",
    "answer_space_problem",
    "rewrite_or_withhold",
]
counts = strict["categorical_results"]["primary_disposition"]
audit_rows = [
    {
        "disposition": label,
        "items": counts[label],
        "share": counts[label] / strict["scope"]["items_reviewed"],
    }
    for label in disposition_order
]
audit_frame = pd.DataFrame(audit_rows)
display(audit_frame.style.format({"share": "{:.1%}"}))

ax = audit_frame.plot.barh(x="disposition", y="items", legend=False, color="#4472C4", figsize=(8, 3.2))
ax.set(xlabel="items (automated strict census)", ylabel="")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

,disposition,items,share
0,usable_as_stored,70,61.9%
1,minor_ui_or_context_change,15,13.3%
2,technically_valid_but_artificial,15,13.3%
3,answer_space_problem,10,8.8%
4,rewrite_or_withhold,3,2.7%


/tmp/ipykernel_627043/509978054.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [4]:
action = cross["agreement"]["action_threshold"]
cross_rows = [
    {"cross-audit outcome": "usable in both audits", "items": action["both_usable"]["count"]},
    {"cross-audit outcome": "action in both audits", "items": action["both_require_action"]["count"]},
    {"cross-audit outcome": "action in strict audit only", "items": action["strict_only_requires_action"]["count"]},
    {"cross-audit outcome": "action in role-specific audit only", "items": action["live_only_requires_action"]["count"]},
]
display(pd.DataFrame(cross_rows))
print(
    f"Coverage: {cross['coverage']['common_items']} items; "
    f"{cross['coverage']['live_judgments']} role judgments across "
    f"{len(cross['coverage']['live_roles'])} roles. "
    f"Union requiring action: {sum(row['items'] for row in cross_rows[1:])}/113."
)

,cross-audit outcome,items
0,usable in both audits,60
1,action in both audits,24
2,action in strict audit only,19
3,action in role-specific audit only,10


Coverage: 113 items; 452 role judgments across 4 roles. Union requiring action: 53/113.


### Exact stored learner-facing examples

These are verbatim prompt/target pairs retained in the compact cross-audit
summary. They illustrate why linguistic validity and learner-facing measurement
validity must be recorded separately.

In [5]:
panels = ["shared_clear_pass", "shared_answer_space_failure", "shared_artificial_interface"]
examples_by_panel = {row["panel"]: row for row in cross["representative_items"]}
example_rows = []
for panel in panels:
    row = examples_by_panel[panel]
    example_rows.append({
        "panel": panel,
        "item_id": row["item_id"],
        "prompt (verbatim)": row["prompt"],
        "target (verbatim)": row["target_answer"],
        "strict disposition": row["strict"]["raw_disposition"],
        "strict learner note": row["strict"]["learner_note"],
    })
display(pd.DataFrame(example_rows))

,panel,item_id,prompt (verbatim),target (verbatim),strict disposition,strict learner note
0,shared_clear_pass,candidate_gc_0397fa37f2228649_02,"The workers finished painting the room before lunch. Complete the sentence using the verb in brackets: By lunchtime, the room _____. (pa...","By lunchtime, the room had been painted.",usable_as_stored,The before-lunch/by-lunchtime contrast and verb cue make the expected span clear.
1,shared_answer_space_failure,candidate_gc_0397fa37f2228649_01,"When we arrived at the park, the workers were gone and the gate was open. Complete the sentence using the cue “open”: The gate ____.",The gate had been opened.,answer_space_problem,“The gate was open” supports an adjectival state or simple-past completion as readily as past-perfect passive “had been opened.”
2,shared_artificial_interface,cue_bounded_imperative_gc_04a854582c08aa84_01,A child reaches toward a hot pan. Give a warning using an ordinary uncontracted negative imperative. Lexical cue chunks (deliberately ou...,Do not touch the hot pan.,technically_valid_but_artificial,"The 78-word rule set is answerable but burdens the learner with “all and only,” function-word, punctuation, and exclusion instructions."


**Audit interpretation.** Seventy of 113 items were usable as stored in the
strict audit, but 43 required at least some action there; the union across both
automated audits flagged 53. The examples show the distinction: a clean cloze,
an underdetermined answer space, and an answerable but annotation-like interface.
This triage motivated a matched-format construction attempt; it does not validate
deployability.

## 2. Outcome-blind KC induction stability

Three independent automated inductions received frozen GrammarCell inputs but no
learner outcomes. Hypotheses are canonicalized by their activation sets, so the
comparison concerns structural behavior rather than wording.

In [6]:
replicate_rows = []
for row in kc["replicates"]:
    replicate_rows.append({
        "replicate": row["replicate_id"],
        "raw hypotheses": row["raw_hypotheses"],
        "unique activations": row["unique_activation_hypotheses"],
        "Q rank": row["unique_q_rank"],
        "exact K* activation matches": row["exact_kstar_match_count"],
        "median support cells": row["support_cells"]["median"],
    })
display(pd.DataFrame(replicate_rows))

agreement_rows = [
    {
        "pair": f"{row['left']} vs {row['right']}",
        "shared activations": row["shared_activation_hypotheses"],
        "union activations": row["union_activation_hypotheses"],
        "activation Jaccard": row["jaccard"],
    }
    for row in kc["pairwise_activation_set_agreement"]
]
display(pd.DataFrame(agreement_rows).style.format({"activation Jaccard": "{:.3f}"}))

pd.DataFrame(agreement_rows).plot.bar(
    x="pair", y="activation Jaccard", legend=False, color="#ED7D31", ylim=(0, 1), figsize=(7, 3)
)
plt.ylabel("activation-set Jaccard")
plt.xlabel("")
plt.tight_layout()
plt.show()
print(
    f"Union={kc['activation_hypotheses_in_union']}; shared by all="
    f"{kc['activation_hypotheses_shared_by_all_replicates']}; "
    f"K* columns exactly recovered by any replicate="
    f"{kc['kstar_columns_recovered_by_any_exact_activation']}/18."
)

,replicate,raw hypotheses,unique activations,Q rank,exact K* activation matches,median support cells
0,independent_01,18,17,17,5,9.0
1,independent_02,18,18,18,4,5.0
2,independent_03,18,18,17,7,9.5


,pair,shared activations,union activations,activation Jaccard
0,independent_01 vs independent_02,11,24,0.458
1,independent_01 vs independent_03,10,25,0.400
2,independent_02 vs independent_03,11,25,0.440


Union=30; shared by all=9; K* columns exactly recovered by any replicate=7/18.


/tmp/ipykernel_627043/1480478974.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**KC interpretation.** Pairwise activation Jaccard is only 0.400–0.458, with
9 hypotheses shared across all runs out of a 30-signature union. Rank can be high
while ontology choice remains unstable. These runs therefore diagnose
underdetermination from GrammarCells alone; they neither select a replacement for
K* nor establish psychological truth.

## 3. Matched-format bank: funnel, geometry, and the failed release gate

The protocol required 38 semantic families × four formats = 152 validated slots,
with the 18 seen cells retaining rank 18. Whole-family acceptance was deliberately
strict: one weak format rejected the family.

In [7]:
round_rows = []
for row in bank["pass_funnel"]["by_candidate_round"]:
    round_rows.append({
        "round": row["candidate_round"],
        "evaluated": row["candidates_evaluated"],
        "deterministic pass": row["deterministic_gate_pass"],
        "solver pass": row["solver_family_gate_pass"],
        "critic/accepted": row["critic_family_gate_pass"],
    })
round_frame = pd.DataFrame(round_rows)
display(round_frame)
round_frame.set_index("round")[["deterministic pass", "solver pass", "critic/accepted"]].plot(
    marker="o", figsize=(7, 3)
)
plt.ylabel("families")
plt.xticks([1, 2, 3])
plt.tight_layout()
plt.show()

g = bank["accepted_family_geometry"]
geometry_rows = [
    {"gate": "families", "observed": g["accepted_family_count"], "required": g["required_families"]},
    {"gate": "format slots", "observed": g["accepted_item_slots"], "required": g["required_item_slots"]},
    {"gate": "selected cells covered", "observed": g["selected_cells_covered"], "required": g["selected_cells_required"]},
    {"gate": "generator KCs covered", "observed": g["active_kcs_covered"], "required": g["generator_kcs_required"]},
    {"gate": "seen Q rank", "observed": g["accepted_seen_q_rank"], "required": g["required_seen_q_rank"]},
]
display(pd.DataFrame(geometry_rows))

,round,evaluated,deterministic pass,solver pass,critic/accepted
0,1,38,31,12,3
1,2,35,32,12,2
2,3,33,26,6,0


/tmp/ipykernel_627043/2956095100.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,gate,observed,required
0,families,5,38
1,format slots,20,152
2,selected cells covered,4,20
3,generator KCs covered,6,18
4,seen Q rank,3,18


### Exact four-format family that passed

This verbatim retained example shows what the intended crossing looked like.
It is **construction evidence only**: five families passed in total, so this is
not a partial dataset release and does not satisfy the bank's coverage/rank gate.

In [8]:
family = bank["accepted_family_geometry"]["accepted_family_examples"][0]
print("Canonical target:", family["canonical_target_sentence"])
print("GrammarCell:", family["cell_id"], "| KCs:", ", ".join(family["generator_kc_ids"]))
family_rows = [
    {
        "format": item["format"],
        "context (verbatim)": item["context"],
        "instruction (verbatim)": item["instruction"],
        "target response (verbatim)": item["target_response"],
    }
    for item in family["learner_facing_items"]
]
display(pd.DataFrame(family_rows))

Canonical target: The delivery might arrive this afternoon.
GrammarCell: gc_90b9229122fa55d6 | KCs: gkc_modal_might


,format,context (verbatim),instruction (verbatim),target response (verbatim)
0,constrained_cloze,"The driver has been delayed, but the delivery is still possible later today.",Complete the sentence using “might” and the correct form of “arrive.”,might arrive
1,dialogue_completion,Nina and Omar are discussing a delayed delivery.,Complete Nina’s reply using “might” and the correct form of “arrive.”,might arrive
2,multiple_choice,"The driver has been delayed, but the delivery is still possible later today.",Choose the sentence that correctly expresses the possibility.,B
3,sentence_transformation,"The driver has been delayed, so the arrival time is uncertain.",Rewrite the sentence using “might” to express the same possibility.,The delivery might arrive this afternoon.


**Bank conclusion.** After three preregistered rounds, only 5/38 families
(20/152 item slots) passed, covering 4/20 selected cells and 6/18 KCs; seen-cell
rank was 3/18. The release gate correctly failed. All downstream simulation
results in this notebook therefore use neutral format labels and Q rows in a
content-free controlled scaffold—not rejected prompts and not a learner-facing
bank.

## 4. Controlled A–D nuisance worlds

The primary estimand is held-out-learner prediction on **seen terminal probes**
(82 learners × 144 slots per seed). Lower log loss/Brier/ECE and lower
item-prerequisite-state RMSE are better. K* is generator truth only inside this
declared simulator.

In [9]:
model_rows = []
for label, row in worlds["model_conditions"].items():
    model_rows.append({
        "model": label,
        "features": row["feature_count"],
        "KC representation": row["kc_representation"],
        "nuisance representation": row["nuisance"],
        "scientific role": row.get("role", "comparison model"),
    })
display(pd.DataFrame(model_rows))

,model,features,KC representation,nuisance representation,scientific role
0,A,20,shared_K_star,none,comparison model
1,B,74,false_format_split_K_star,implicit_format_through_split_history_and_indicators,comparison model
2,C,23,shared_K_star,three_observed_format_contrasts,comparison model
3,D,146,shared_K_star,format contrasts plus 123-dimensional Q*/format-orthogonal residual basis for 144 acquisition-seen item slots; eight probe-only items ze...,oracle_aligned_same_seen_item_positive_control


In [10]:
world_order = [
    "clean_zero", "format_moderate", "format_strong_control",
    "item_moderate", "item_format_moderate", "combined_heterogeneous",
]
abcd_rows = []
for world_id in world_order:
    for model in "ABCD":
        metrics = worlds["abcd_seen_terminal_probe_summary"]["models"][world_id][model]["across_seed"]
        abcd_rows.append({
            "world": world_id,
            "model": model,
            "log loss": metrics["log_loss"]["mean"],
            "Brier": metrics["brier_score"]["mean"],
            "ECE": metrics["ece_10_fixed_width"]["mean"],
            "state RMSE": metrics["item_prerequisite_state_rmse"]["mean"],
        })
abcd_frame = pd.DataFrame(abcd_rows)
display(abcd_frame.style.format({"log loss": "{:.6f}", "Brier": "{:.6f}", "ECE": "{:.6f}", "state RMSE": "{:.6f}"}))

pivot = abcd_frame.pivot(index="world", columns="model", values="log loss").loc[world_order]
pivot.plot(marker="o", figsize=(10, 4))
plt.ylabel("seen-probe log loss (3-seed mean)")
plt.xlabel("")
plt.xticks(range(len(world_order)), world_order, rotation=25, ha="right")
plt.tight_layout()
plt.show()

,world,model,log loss,Brier,ECE,state RMSE
0,clean_zero,A,0.657219,0.232512,0.020034,0.115625
1,clean_zero,B,0.664138,0.235802,0.012487,0.135821
2,clean_zero,C,0.657231,0.232517,0.020041,0.115624
3,clean_zero,D,0.657843,0.232807,0.020121,0.115743
4,format_moderate,A,0.659499,0.233599,0.020558,0.117301
5,format_moderate,B,0.657328,0.232559,0.013345,0.154992
6,format_moderate,C,0.651279,0.229697,0.021077,0.116379
7,format_moderate,D,0.651795,0.229938,0.021019,0.116483
8,format_strong_control,A,0.663597,0.235550,0.021078,0.122492
9,format_strong_control,B,0.638965,0.223923,0.013055,0.201946


/tmp/ipykernel_627043/3569538278.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [11]:
contrast_rows = []
for contrast, row in worlds["contrasts"]["primary_cross_world"].items():
    summary = row["across_seed_point_estimate"]
    exclusions = sum(
        interval["percentile_95"][1] < 0 or interval["percentile_95"][0] > 0
        for interval in row["per_seed"].values()
    )
    contrast_rows.append({
        "contrast": contrast,
        "mean delta log loss": summary["mean"],
        "seed range": f"[{summary['minimum']:.6f}, {summary['maximum']:.6f}]",
        "seed-conditional intervals excluding 0": f"{exclusions}/3",
        "correct sign interpretation": row["corrected_sign_gloss"],
    })
display(pd.DataFrame(contrast_rows).style.format({"mean delta log loss": "{:.6f}"}))

,contrast,mean delta log loss,seed range,seed-conditional intervals excluding 0,correct sign interpretation
0,explicit_format_remedy,-0.005317,"[-0.006034, -0.004652]",3/3,Negative means shared-K* model C with an explicit observed format covariate predicts better than false format-split model B in the strong planted-format control world.
1,explicit_item_remedy_combined,-0.012609,"[-0.013506, -0.011947]",3/3,Negative means model D recovers deliberately planted stable effects for acquisition-seen item IDs better than C in the item-plus-format world.
2,explicit_item_remedy_item_only,-0.013099,"[-0.013313, -0.012698]",3/3,Negative means model D recovers deliberately planted stable effects for acquisition-seen item IDs better than C in the item-only world.
3,format_confounding_difference_in_differences,-0.031551,"[-0.033431, -0.029013]",3/3,Negative means planted format nuisance increases false format-split model B's predictive advantage relative to shared-K* model A. It does not mean that an explicitly corrected model wins and does not validate B as a psychological ontology.


In [12]:
def interval_rows(label, payload):
    rows = []
    for seed, result in payload["per_seed"].items():
        lo, hi = result["percentile_95"]
        rows.append({
            "control": label,
            "seed": seed,
            "delta log loss": result["point_estimate"],
            "conditional 95% interval": f"[{lo:.6f}, {hi:.6f}]",
            "relation to zero": "below" if hi < 0 else "above" if lo > 0 else "contains",
        })
    return rows

sensitivity_rows = []
sensitivity_rows += interval_rows("item-only B−A", worlds["contrasts"]["item_only_false_split_B_minus_A"])
sensitivity_rows += interval_rows("heterogeneous C−B", worlds["contrasts"]["combined_heterogeneous"]["C_minus_B"])
sensitivity_rows += interval_rows("heterogeneous D−C", worlds["contrasts"]["combined_heterogeneous"]["D_minus_C"])
display(pd.DataFrame(sensitivity_rows).style.format({"delta log loss": "{:+.6f}"}))

,control,seed,delta log loss,conditional 95% interval,relation to zero
0,item-only B−A,20260829,+0.001195,"[-0.001665, 0.004110]",contains
1,item-only B−A,20260830,+0.000847,"[-0.001477, 0.003183]",contains
2,item-only B−A,20260831,-0.001328,"[-0.003652, 0.001060]",contains
3,heterogeneous C−B,20260829,-0.002226,"[-0.005138, 0.000695]",contains
4,heterogeneous C−B,20260830,+0.000169,"[-0.002073, 0.002427]",contains
5,heterogeneous C−B,20260831,+0.001074,"[-0.001241, 0.003341]",contains
6,heterogeneous D−C,20260829,-0.010738,"[-0.013911, -0.007473]",below
7,heterogeneous D−C,20260830,-0.013005,"[-0.015793, -0.010294]",below
8,heterogeneous D−C,20260831,-0.012350,"[-0.015673, -0.009205]",below


**A–D interpretation.** The format difference-in-differences is −0.03155:
the correct sign gloss is that planted format nuisance increases false-split B's
relative predictive advantage over shared-K* A. It does **not** validate B as a
psychological ontology. Explicit format model C beats B in the strong control,
but that remedy is mixed under combined heterogeneity (all three intervals contain
zero). D beats C under planted item effects, but D is an **oracle-aligned,
same-seen-item positive control**: the planted effect lies in its 123-dimensional
seen-item residual basis and held-out items are zero encoded. This is not evidence
that an arbitrary real-data item model will deconfound unseen items.

All intervals are 2,000-repeat learner-cluster percentile bootstraps over fixed
held-out predictions. They cover test-learner variation conditional on the frozen
fit; they exclude train/dev sampling, refitting/tuning, item-bank sampling,
simulator/world uncertainty, and seed uncertainty. Three-seed means/ranges are
descriptive, not confidence intervals.

## 5. Structured synthetic error histories

The same binary outcomes are augmented with linked, 80%-linked, or within-item
shuffled categories. The target is a **post-outcome deficit-proportional
attribution**, not a causal human error diagnosis. Prediction uses only prior
categories; current-response labels are not leaked.

In [13]:
stream_order = [
    "binary_only", "linked_positive_control", "linked_80_percent",
    "within_item_shuffled_negative_control",
]
error_rows = []
for stream in stream_order:
    pred = worlds["error_history"]["prediction_and_item_prerequisite_state"][stream]["across_seed"]
    loc = worlds["error_history"]["failed_kc_localisation"][stream]
    terminal = worlds["error_history"]["secondary_terminal_kc_evidence_diagnostic"][stream]
    error_rows.append({
        "history": stream,
        "prediction log loss": pred["log_loss"]["mean"],
        "prediction Brier": pred["brier_score"]["mean"],
        "item-state RMSE": pred["item_prerequisite_state_rmse"]["mean"],
        "failed-KC compatible top-1": loc["across_seed"]["compatible_top1"]["mean"],
        "terminal KC evidence RMSE": terminal["across_seed"]["rmse"]["mean"],
    })
error_frame = pd.DataFrame(error_rows)
display(error_frame.style.format({column: "{:.6f}" for column in error_frame.columns if column != "history"}))

,history,prediction log loss,prediction Brier,item-state RMSE,failed-KC compatible top-1,terminal KC evidence RMSE
0,binary_only,0.636359,0.222758,0.129724,0.420781,0.228727
1,linked_positive_control,0.635493,0.222362,0.125210,1.000000,0.144357
2,linked_80_percent,0.635833,0.222510,0.127232,0.883728,0.158804
3,within_item_shuffled_negative_control,0.636987,0.223034,0.131138,0.462525,0.165519


In [14]:
paired_rows = []
for comparison, payload in worlds["error_history"]["paired_prediction_log_loss"].items():
    for seed, result in payload["per_seed"].items():
        lo, hi = result["percentile_95"]
        paired_rows.append({
            "comparison": comparison,
            "seed": seed,
            "delta log loss": result["point_estimate"],
            "conditional 95% interval": f"[{lo:.6f}, {hi:.6f}]",
            "relation to zero": "below" if hi < 0 else "above" if lo > 0 else "contains",
        })
display(pd.DataFrame(paired_rows).style.format({"delta log loss": "{:+.6f}"}))

,comparison,seed,delta log loss,conditional 95% interval,relation to zero
0,linked_80_percent_minus_binary_only,20260829,-0.000902,"[-0.001716, -0.000123]",below
1,linked_80_percent_minus_binary_only,20260830,-0.000469,"[-0.001334, 0.000384]",contains
2,linked_80_percent_minus_binary_only,20260831,-0.000209,"[-0.001041, 0.000545]",contains
3,linked_positive_control_minus_binary_only,20260829,-0.001336,"[-0.002358, -0.000304]",below
4,linked_positive_control_minus_binary_only,20260830,-0.000897,"[-0.001822, -0.000008]",below
5,linked_positive_control_minus_binary_only,20260831,-0.000368,"[-0.001247, 0.000544]",contains
6,within_item_shuffled_negative_control_minus_binary_only,20260829,+0.000689,"[-0.000105, 0.001503]",contains
7,within_item_shuffled_negative_control_minus_binary_only,20260830,+0.000608,"[-0.000111, 0.001287]",contains
8,within_item_shuffled_negative_control_minus_binary_only,20260831,+0.000587,"[-0.000310, 0.001438]",contains


**Error interpretation.** Fully linked categories are a positive control and
localize the synthetic target perfectly by construction; 80%-linked categories
remain strongly diagnostic. Their next-response log-loss gains are small and not
uniform across seeds, so a predictive benefit is not consistently established.
Within-item shuffling is an event-link negative control—not an information-free
chance null—because it preserves item/category marginals. Its localization log
loss is also on a different task/scale from response prediction and must not be
compared numerically. The terminal-KC RMSE is a separate model-independent
Beta(1,1) evidence-count diagnostic, not an A–D fitted state and not human mastery.

## 6. Exploratory schedule-policy recovery

These post-response derived analyses apply the frozen A–D fitting protocol to
alternative controlled histories. They are explicitly exploratory
recovery/model-fit comparisons—not policy efficacy estimates.

In [15]:
policy_order = ["q_balanced_lab", "curriculum", "mixed_practice", "adaptive_weakness"]
policy_rows = []
for name in policy_order:
    result = policy["policy_results"][name]
    schedule = worlds["schedule_diagnostics"]["policies"][name]["across_seed"]
    policy_rows.append({
        "policy": name,
        "D seen log loss": result["condition_D_seen_log_loss"]["mean"],
        "D item-state RMSE": result["condition_D_seen_item_prerequisite_state_rmse"]["mean"],
        "binary terminal-KC evidence RMSE": result["binary_terminal_Kstar_evidence_rmse"]["mean"],
        "item exposure Gini": schedule["item_exposure_gini"]["mean"],
        "median item repetition gap": schedule["median_repetition_gap"]["mean"],
        "mean adjacent-Q Jaccard": schedule["mean_adjacent_q_jaccard_design_linked"]["mean"],
    })
policy_frame = pd.DataFrame(policy_rows)
display(policy_frame.style.format({column: "{:.6f}" for column in policy_frame.columns if column != "policy"}))

comparison_rows = []
for name, payload in policy["learner_paired_seen_log_loss_comparisons"].items():
    summary = payload["point_estimate_summary"]
    excluded = sum(
        result["percentile_95"][1] < 0 or result["percentile_95"][0] > 0
        for result in payload["per_seed"].values()
    )
    comparison_rows.append({
        "comparison": name,
        "mean policy−q-balanced log loss": summary["mean"],
        "seed range": f"[{summary['minimum']:.6f}, {summary['maximum']:.6f}]",
        "seed-conditional intervals excluding 0": f"{excluded}/3",
    })
display(pd.DataFrame(comparison_rows).style.format({"mean policy−q-balanced log loss": "{:+.6f}"}))

,policy,D seen log loss,D item-state RMSE,binary terminal-KC evidence RMSE,item exposure Gini,median item repetition gap,mean adjacent-Q Jaccard
0,q_balanced_lab,0.636359,0.129724,0.228727,0.162530,93.666667,0.111782
1,curriculum,0.639779,0.139472,0.229618,0.162530,31.000000,0.260522
2,mixed_practice,0.636341,0.130748,0.229042,0.162530,92.000000,0.059951
3,adaptive_weakness,0.639477,0.141659,0.237280,0.080298,26.666667,0.348607


,comparison,mean policy−q-balanced log loss,seed range,seed-conditional intervals excluding 0
0,adaptive_weakness_minus_q_balanced_lab,+0.003118,"[0.002572, 0.004113]",1/3
1,curriculum_minus_q_balanced_lab,+0.003420,"[0.002595, 0.004051]",2/3
2,mixed_practice_minus_q_balanced_lab,-0.000018,"[-0.001020, 0.001574]",0/3


**Schedule interpretation.** Mixed practice is nearly null against q-balanced
(mean Δ log loss −0.000018; 0/3 seed-conditional intervals exclude zero).
Curriculum and adaptive point means are +0.003420 and +0.003118, but exclusions
occur in only 2/3 and 1/3 seeds. Q-balanced, curriculum, and mixed use the same
188-event multiset and have identical terminal oracle mastery by construction;
adaptive changes exposure/state and therefore mixes mechanisms. The schedule
columns characterize exposure, spacing, and interleaving only. No policy ranking,
causal efficacy claim, or real-platform claim is permitted.

## 7. Ecological-realism / measurement-precision continuum

Four matched GrammarCell families were rendered at five openness levels and
assessed by five automated critic roles (20 opportunities, 100 judgments). The
separate diagnostics are retained without a composite “realism score.”

In [16]:
format_order = dialogue["format_order"]
dialogue_rows = []
for name in format_order:
    result = dialogue["by_format"][name]
    ratings = result["rating_distributions"]
    dialogue_rows.append({
        "format": name,
        "determinate": ratings["answer_determinacy"].get("determinate", 0),
        "KC clear": ratings["kc_attribution"].get("clear", 0),
        "response-family lower bound (mean)": result["plausible_response_lower_bound"]["mean"],
        "interaction naturalness pass": ratings["interaction_naturalness"].get("pass", 0),
        "target-avoiding shortcut": result["target_avoiding_shortcut"]["true"],
        "incidental operations (mean)": result["incidental_grammar"]["count_per_judgment"]["mean"],
    })
dialogue_frame = pd.DataFrame(dialogue_rows)
display(dialogue_frame)

rates = dialogue_frame.set_index("format")[["determinate", "KC clear", "interaction naturalness pass"]] / 20
rates.plot(marker="o", figsize=(9, 3.5), ylim=(0, 1.05))
plt.ylabel("share of 20 automated judgments")
plt.xlabel("")
plt.xticks(range(len(format_order)), format_order, rotation=20, ha="right")
plt.tight_layout()
plt.show()

,format,determinate,KC clear,response-family lower bound (mean),interaction naturalness pass,target-avoiding shortcut,incidental operations (mean)
0,constrained_cloze,17,17,1.30,17,1,1.25
1,sentence_transformation,4,10,2.70,13,6,2.10
2,contextual_production,1,5,3.50,10,8,1.90
3,dialogue_completion,0,3,3.85,16,14,2.30
4,open_dialogue,0,4,4.55,20,13,2.75


/tmp/ipykernel_627043/3858094186.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [17]:
open_deltas = dialogue["matched_deltas_vs_constrained_cloze"]["open_dialogue"]["separate_metric_deltas_target_minus_reference"]
selected_delta_labels = [
    "determinacy_risk", "kc_attribution_risk", "plausible_response_lower_bound",
    "incidental_grammar_count", "target_avoiding_shortcut", "interaction_naturalness_risk",
]
display(pd.DataFrame([
    {
        "open dialogue − constrained cloze diagnostic": label,
        "matched mean delta": open_deltas[label]["mean"],
        "matched comparisons": open_deltas[label]["count"],
    }
    for label in selected_delta_labels
]))

,open dialogue − constrained cloze diagnostic,matched mean delta,matched comparisons
0,determinacy_risk,1.00,19
1,kc_attribution_risk,0.70,20
2,plausible_response_lower_bound,3.25,20
3,incidental_grammar_count,1.50,20
4,target_avoiding_shortcut,0.60,20
5,interaction_naturalness_risk,-0.15,20


**Continuum interpretation.** Constrained cloze was determinate and KC-clear in
17/20 judgments; open dialogue was determinate in 0/20 and KC-clear in 4/20, while
passing interaction naturalness in 20/20. Relative to cloze, open dialogue raised
the plausible response-family lower bound by 3.25 and incidental-operation count
by 1.50, while reducing naturalness risk by 0.15. This is an automated stress test
of the ecological-realism/measurement-precision tradeoff—not human response-
process evidence and not proof that an interface is deployable.

## 8. Evidence ledger and bounded conclusion

In [18]:
MEASUREMENT_REALISM_RESULTS = {
    "platform_audit": {
        "strict_usable_as_stored": counts["usable_as_stored"],
        "strict_items": strict["scope"]["items_reviewed"],
        "cross_audit_union_requiring_action": sum(row["items"] for row in cross_rows[1:]),
    },
    "kc_induction": {
        "activation_union": kc["activation_hypotheses_in_union"],
        "activation_shared_all_three": kc["activation_hypotheses_shared_by_all_replicates"],
        "pairwise_jaccard_range": [
            min(row["jaccard"] for row in kc["pairwise_activation_set_agreement"]),
            max(row["jaccard"] for row in kc["pairwise_activation_set_agreement"]),
        ],
    },
    "matched_bank": {
        "families_passed": g["accepted_family_count"],
        "families_required": g["required_families"],
        "seen_rank": g["accepted_seen_q_rank"],
        "required_seen_rank": g["required_seen_q_rank"],
        "release": False,
    },
    "controlled_worlds": {
        "format_DiD_mean": worlds["contrasts"]["primary_cross_world"]["format_confounding_difference_in_differences"]["across_seed_point_estimate"]["mean"],
        "item_positive_control_D_minus_C_mean": worlds["contrasts"]["primary_cross_world"]["explicit_item_remedy_item_only"]["across_seed_point_estimate"]["mean"],
        "release_eligible": worlds["release_eligible"],
    },
    "validation": {
        "human_validation": False,
        "learner_facing_measurement_validity": "NOT_ASSESSED_FOR_CONTROLLED_WORLDS",
        "platform_plausibility": "NOT_ASSESSED_FOR_CONTROLLED_WORLDS",
        "new_dataset_release": False,
    },
}
MEASUREMENT_REALISM_RESULTS

{'platform_audit': {'strict_usable_as_stored': 70,
  'strict_items': 113,
  'cross_audit_union_requiring_action': 53},
 'kc_induction': {'activation_union': 30,
  'activation_shared_all_three': 9,
  'pairwise_jaccard_range': [0.4, 0.4583333333333333]},
 'matched_bank': {'families_passed': 5,
  'families_required': 38,
  'seen_rank': 3,
  'required_seen_rank': 18,
  'release': False},
 'controlled_worlds': {'format_DiD_mean': -0.03155148583847481,
  'item_positive_control_D_minus_C_mean': -0.013098762124398365,
  'release_eligible': False},
 'validation': {'human_validation': False,
  'learner_facing_measurement_validity': 'NOT_ASSESSED_FOR_CONTROLLED_WORLDS',
  'platform_plausibility': 'NOT_ASSESSED_FOR_CONTROLLED_WORLDS',
  'new_dataset_release': False}}

### Bounded conclusion

The retained evidence supports a methodological claim: in a known-truth
synthetic environment, measurement nuisance can make an incorrect KC split look
predictively useful, explicit nuisance controls can recover planted effects under
their declared controls, and structured error categories can preserve diagnostic
information discarded by correctness alone. It also exposes hard limits: KC
induction from GrammarCells is unstable, the matched learner-facing bank did not
pass its release gate, and increasing interaction openness traded measurement
precision for naturalness in automated critique.

> **Final boundary:** this is a controlled-scenario results notebook, not a new
> dataset release. Its learner-facing examples are audit/construction evidence,
> not a validated bank. It contains no human validation and supports no claim
> that the scaffold is a realistic platform dataset or that its simulator
> parameters describe human learners.